In [1]:
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob
from datetime import timedelta
import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"


In [2]:
# Load trip data
citibike_data_path = "/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv"

# Read the CSV file into a pandas DataFrame
citibike_data = pd.read_csv(citibike_data_path)

/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_72128/1978427239.py:5: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  citibike_data = pd.read_csv(citibike_data_path)


In [3]:
# Aggregate pairwise trips
pairwise_data = citibike_data.groupby(['start_station_id', 'end_station_id']).agg(
    observed_trips=('ride_id', 'count'),
    origin_activity=('start_station_id', 'size'),  # Total trips originating from each station
    destination_activity=('end_station_id', 'size')  # Total trips ending at each station
).reset_index()

# Calculate distance (Euclidean as an example)
stations = citibike_data[['start_station_id', 'start_lat', 'start_lng']].drop_duplicates()
stations.rename(columns={'start_station_id': 'station_id', 'start_lat': 'lat', 'start_lng': 'lng'}, inplace=True)

# Merge coordinates for origins and destinations
pairwise_data = pairwise_data.merge(
    stations.rename(columns={'station_id': 'start_station_id', 'lat': 'origin_lat', 'lng': 'origin_lng'}),
    on='start_station_id', how='left'
)
pairwise_data = pairwise_data.merge(
    stations.rename(columns={'station_id': 'end_station_id', 'lat': 'destination_lat', 'lng': 'destination_lng'}),
    on='end_station_id', how='left'
)

# Calculate Euclidean distance
import numpy as np
pairwise_data['distance'] = np.sqrt(
    (pairwise_data['origin_lat'] - pairwise_data['destination_lat'])**2 +
    (pairwise_data['origin_lng'] - pairwise_data['destination_lng'])**2
)


In [4]:
from sklearn.linear_model import LinearRegression

# Add logarithmic transformations
pairwise_data['log_observed_trips'] = np.log(pairwise_data['observed_trips'])
pairwise_data['log_origin_activity'] = np.log(pairwise_data['origin_activity'])
pairwise_data['log_destination_activity'] = np.log(pairwise_data['destination_activity'])
pairwise_data['log_distance'] = np.log(pairwise_data['distance'])

# Prepare regression variables
X = pairwise_data[['log_origin_activity', 'log_destination_activity', 'log_distance']]
y = pairwise_data['log_observed_trips']

# Fit the linear regression model
model = LinearRegression()
model.fit(X, y)

# Extract coefficients
log_k = model.intercept_
beta_origin, beta_destination, beta_distance = model.coef_

# Display model parameters
k = np.exp(log_k)
print(f"Scaling Factor (k): {k}")
print(f"Origin Activity Coefficient (beta_origin): {beta_origin}")
print(f"Destination Activity Coefficient (beta_destination): {beta_destination}")
print(f"Distance Decay Coefficient (beta_distance): {-beta_distance}")


/opt/anaconda3/envs/ox/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [5]:
# Predict trips using the gravity model
pairwise_data['predicted_trips'] = k * (
    pairwise_data['origin_activity'] ** beta_origin *
    pairwise_data['destination_activity'] ** beta_destination /
    pairwise_data['distance'] ** beta_distance
)

# Compare predicted trips with observed trips
print(pairwise_data[['start_station_id', 'end_station_id', 'observed_trips', 'predicted_trips']].head())


NameError: name 'k' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of observed vs. predicted trips
plt.figure(figsize=(8, 6))
plt.scatter(pairwise_data['observed_trips'], pairwise_data['predicted_trips'], alpha=0.5)
plt.plot([0, max(pairwise_data['observed_trips'])], [0, max(pairwise_data['predicted_trips'])], color='red', linestyle='--')
plt.xlabel('Observed Trips')
plt.ylabel('Predicted Trips')
plt.title('Gravity Model: Observed vs. Predicted Trips')
plt.show()
